# SparkOCR-VLM — Evaluation

Score pipeline output against committed goldens. Uses mock backend so the notebook is offline-safe.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve() / 'src'))

from pathlib import Path
from tests.harness.synthetic_pdf import SyntheticPDFBuilder
from tests.harness.golden import install_mock_table_from_fixtures

FIX = Path('../tests/fixtures')
SyntheticPDFBuilder(out_dir=FIX).build_all()
install_mock_table_from_fixtures(FIX)

In [ ]:
from sparkocr_vlm import OCRPipeline
pipeline = OCRPipeline(backend='mock')

import pandas as pd
rows = []
for p in sorted(FIX.glob('synth_*.pdf')):
    rows.append(pipeline.parse_single(p))
df = pd.concat(rows, ignore_index=True)
df[['filename','page_num','markdown']].head()

In [ ]:
from sparkocr_vlm.evaluator import Evaluator
ev = Evaluator(ground_truth_dir=FIX / 'golden', anchors_by_doc={'synth_invoice': ['INV-2024-001', '$1,234.56']})
metrics = ev.score_df(df)
metrics

## Bar chart of per-doc edit distance

In [ ]:
import matplotlib.pyplot as plt
from sparkocr_vlm.evaluator import normalized_edit_distance

dists = []
labels = []
for _, row in df.iterrows():
    gt_path = FIX / 'golden' / (Path(row['filename']).stem + '.md')
    if gt_path.exists():
        dists.append(normalized_edit_distance(row['markdown'], gt_path.read_text()))
        labels.append(f"{row['filename']} p{row['page_num']}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(dists)), dists)
ax.set_xticks(range(len(dists)))
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('Normalized edit distance')
ax.set_title('Per-doc OCR error')
plt.tight_layout()